# 03g — Exp 7: Ablation Table

**Project:** UREP 32-0210-250078 | Crack Classification

Loads all experiment checkpoints, evaluates each on the test set, and builds the
ablation table that quantifies each component's contribution.

| Config | Accuracy | Macro-F1 | Shear Recall | hP | hR | hF |
|--------|----------|----------|-------------|----|----|----|
| Baseline (02i, multix2) | | | | | | |
| + Logit Adjust (Exp 1) | | | | | | |
| + τ-Norm (Exp 2) | | | | | | |
| + CB Focal Loss (Exp 3) | | | | | | |
| + cRT (Exp 4) | | | | | | |
| + Pairwise Confusion (Exp 5) | | | | | | |
| + Confusion OS (Exp 6) | | | | | | |

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import json
import copy
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, recall_score, accuracy_score, classification_report

import config
from src.device import print_device_summary, get_device, set_seed
from src.model_cbam_hierarchical import InceptionV3CBAMHierarchical
from src.hierarchical import (
    STAGE1_CLASSES, STAGE2_CLASSES, STAGE3_CLASSES,
    get_hierarchical_dataloaders,
    evaluate_hierarchical_model,
    hierarchical_pr_f1, per_stage_confusion_matrices, error_attribution,
)
from src.losses import (
    compute_class_frequencies,
    evaluate_hierarchical_with_logit_adj,
    tau_normalize_hierarchical,
    HierarchicalCrackDataset,
)
from src.augmentation import get_val_test_transforms

set_seed(config.RANDOM_SEED)

OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "ablation")
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

device_config = print_device_summary()
device = get_device()
BATCH = device_config["batch_sizes"][1]
NUM_WORKERS = device_config["num_workers"]

print(f"\nExperiment 7: Ablation Table")
print(f"Device: {device}")

## Define experiment registry

In [ ]:
experiments = {
    "Baseline (02i, multix2)": {
        "ckpt": os.path.join(config.OUTPUT_DIR, "cbam_hier_multix2", "models", "best_model.pt"),
        "mode": "standard",
    },
    "+ Logit Adj (Exp 1)": {
        "ckpt": os.path.join(config.OUTPUT_DIR, "cbam_hier_multix2", "models", "best_model.pt"),
        "mode": "logit_adj",
        "results_json": os.path.join(config.OUTPUT_DIR, "exp1_logit_adj", "exp1_results.json"),
    },
    "+ tau-Norm (Exp 2)": {
        "ckpt": os.path.join(config.OUTPUT_DIR, "exp2_tau_norm", "models", "best_model.pt"),
        "mode": "standard",
        "fallback_ckpt": os.path.join(config.OUTPUT_DIR, "cbam_hier_multix2", "models", "best_model.pt"),
        "results_json": os.path.join(config.OUTPUT_DIR, "exp2_tau_norm", "exp2_results.json"),
    },
    "+ CB Focal (Exp 3)": {
        "ckpt": os.path.join(config.OUTPUT_DIR, "exp3_cb_focal", "models", "best_model.pt"),
        "mode": "standard",
    },
    "+ cRT (Exp 4)": {
        "ckpt": os.path.join(config.OUTPUT_DIR, "exp4_crt", "models", "best_model.pt"),
        "mode": "standard",
    },
    "+ PC Reg (Exp 5)": {
        "ckpt": os.path.join(config.OUTPUT_DIR, "exp5_pc_reg", "models", "best_model.pt"),
        "mode": "standard",
    },
    "+ Conf. OS (Exp 6)": {
        "ckpt": os.path.join(config.OUTPUT_DIR, "exp6_conf_oversample", "models", "best_model.pt"),
        "mode": "standard",
    },
}

# Check which checkpoints exist
for name, info in experiments.items():
    exists = os.path.exists(info["ckpt"])
    print(f"  {'OK' if exists else 'MISSING':>7}  {name}  [{info['ckpt']}]")

## Build test loader and prepare logit-adj data

In [ ]:
_, _, test_loader = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=BATCH,
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="stage1",
)

# Compute log-frequencies for logit adjustment
train_ds = HierarchicalCrackDataset(
    config.SPLIT_DIR, "train",
    transform=get_val_test_transforms(config.IMG_SIZE, "imagenet"),
)
freqs = compute_class_frequencies(train_ds)
log_freqs = {
    k: torch.tensor(np.log(v + 1e-8), dtype=torch.float32)
    for k, v in freqs.items()
}
del train_ds

## Evaluate all experiments

In [ ]:
shear_idx = config.CLASS_NAMES.index("shear")
all_results = []

for name, info in experiments.items():
    ckpt_path = info["ckpt"]
    if not os.path.exists(ckpt_path):
        # Try fallback
        ckpt_path = info.get("fallback_ckpt", ckpt_path)
        if not os.path.exists(ckpt_path):
            print(f"SKIP {name}: checkpoint not found")
            continue

    print(f"\nEvaluating: {name}")
    model = InceptionV3CBAMHierarchical().to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()

    if info["mode"] == "logit_adj":
        # Load best tau from Exp 1 results
        rj = info.get("results_json", "")
        if os.path.exists(rj):
            with open(rj) as f:
                best_tau = json.load(f)["best_tau"]
        else:
            best_tau = 1.0  # default
        results = evaluate_hierarchical_with_logit_adj(
            model, test_loader, device, log_freqs,
            tau=best_tau, t1=0.5, t2=0.5,
        )
    elif info["mode"] == "tau_norm":
        rj = info.get("results_json", "")
        if os.path.exists(rj):
            with open(rj) as f:
                best_tau = json.load(f)["best_tau"]
        else:
            best_tau = 0.5
        tau_normalize_hierarchical(model, best_tau, best_tau, best_tau)
        results = evaluate_hierarchical_model(model, test_loader, device, t1=0.5, t2=0.5)
    else:
        results = evaluate_hierarchical_model(model, test_loader, device, t1=0.5, t2=0.5)

    yt = np.array([config.CLASS_NAMES.index(c) for c in results["y_true_flat"]])
    yp = np.array([config.CLASS_NAMES.index(c) for c in results["y_pred_flat"]])

    acc = accuracy_score(yt, yp)
    macro_f1 = f1_score(yt, yp, average="macro")
    weighted_f1 = f1_score(yt, yp, average="weighted")
    per_class_recall = recall_score(yt, yp, average=None)
    per_class_f1 = f1_score(yt, yp, average=None)

    h_met = hierarchical_pr_f1(results["y_true_paths"], results["y_pred_paths"])
    err = error_attribution(results["y_true_paths"], results["y_pred_paths"])

    row = {
        "Config": name,
        "Accuracy": acc,
        "Macro-F1": macro_f1,
        "Weighted-F1": weighted_f1,
        "Shear Recall": per_class_recall[shear_idx],
        "Shear F1": per_class_f1[shear_idx],
        "hP": h_met["hP"],
        "hR": h_met["hR"],
        "hF": h_met["hF"],
        "Stage1 Errors": err["errors_by_stage"]["stage1"],
        "Stage2 Errors": err["errors_by_stage"]["stage2"],
        "Stage3 Errors": err["errors_by_stage"]["stage3"],
    }

    # Per-class F1 breakdown
    for i, cls_name in enumerate(config.CLASS_NAMES):
        row[f"F1_{cls_name}"] = per_class_f1[i]

    all_results.append(row)
    print(f"  Acc={acc:.4f}  Macro-F1={macro_f1:.4f}  Shear Recall={per_class_recall[shear_idx]:.4f}")

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Ablation Table

In [ ]:
df = pd.DataFrame(all_results)

# Main table
main_cols = ["Config", "Accuracy", "Macro-F1", "Weighted-F1", "Shear Recall", "Shear F1", "hP", "hR", "hF"]
df_main = df[main_cols].copy()

# Format numbers
for col in main_cols[1:]:
    df_main[col] = df_main[col].map(lambda x: f"{x:.4f}")

print("="*100)
print("ABLATION TABLE")
print("="*100)
print(df_main.to_string(index=False))
print()

## Per-class F1 breakdown

In [ ]:
f1_cols = ["Config"] + [f"F1_{c}" for c in config.CLASS_NAMES]
df_f1 = df[f1_cols].copy()

for col in f1_cols[1:]:
    df_f1[col] = df_f1[col].map(lambda x: f"{x:.4f}")

print("Per-class F1 scores:")
print(df_f1.to_string(index=False))

## Error attribution breakdown

In [ ]:
err_cols = ["Config", "Stage1 Errors", "Stage2 Errors", "Stage3 Errors"]
df_err = df[err_cols].copy()
print("\nError attribution (count of errors per stage):")
print(df_err.to_string(index=False))

## Visualization: Macro-F1 comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart: Macro-F1
configs = [r["Config"] for r in all_results]
macro_f1s = [r["Macro-F1"] for r in all_results]
shear_recalls = [r["Shear Recall"] for r in all_results]

x = np.arange(len(configs))
axes[0].barh(x, macro_f1s, color="steelblue", alpha=0.8)
axes[0].set_yticks(x)
axes[0].set_yticklabels(configs, fontsize=9)
axes[0].set_xlabel("Macro-F1")
axes[0].set_title("Macro-F1 Comparison")
axes[0].axvline(x=macro_f1s[0], color="red", linestyle="--", alpha=0.5, label="baseline")
for i, v in enumerate(macro_f1s):
    axes[0].text(v + 0.002, i, f"{v:.4f}", va="center", fontsize=8)
axes[0].legend()

# Bar chart: Shear Recall
axes[1].barh(x, shear_recalls, color="coral", alpha=0.8)
axes[1].set_yticks(x)
axes[1].set_yticklabels(configs, fontsize=9)
axes[1].set_xlabel("Shear Recall")
axes[1].set_title("Shear Recall Comparison")
axes[1].axvline(x=shear_recalls[0], color="red", linestyle="--", alpha=0.5, label="baseline")
for i, v in enumerate(shear_recalls):
    axes[1].text(v + 0.002, i, f"{v:.4f}", va="center", fontsize=8)
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "ablation_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## Per-class F1 heatmap

In [ ]:
f1_matrix = np.array([[r[f"F1_{c}"] for c in config.CLASS_NAMES] for r in all_results])

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    f1_matrix, annot=True, fmt=".3f", cmap="YlGnBu",
    xticklabels=config.CLASS_NAMES,
    yticklabels=configs,
    ax=ax,
)
ax.set_title("Per-class F1 across experiments")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "per_class_f1_heatmap.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## Save results

In [ ]:
# Save as JSON
with open(os.path.join(OUTPUT_DIR, "ablation_results.json"), "w") as f:
    json.dump(all_results, f, indent=2)

# Save as CSV
df.to_csv(os.path.join(OUTPUT_DIR, "ablation_table.csv"), index=False)

# Save LaTeX table
latex_df = df[main_cols].copy()
for col in main_cols[1:]:
    latex_df[col] = latex_df[col].map(lambda x: f"{x:.4f}" if isinstance(x, float) else x)
latex_str = latex_df.to_latex(index=False, escape=False, column_format="l" + "c" * (len(main_cols) - 1))
with open(os.path.join(OUTPUT_DIR, "ablation_table.tex"), "w") as f:
    f.write(latex_str)

print(f"Results saved to {OUTPUT_DIR}/")
print(f"  - ablation_results.json")
print(f"  - ablation_table.csv")
print(f"  - ablation_table.tex")